In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, lower, count, rank, row_number
from pyspark.sql.window import Window

spark = spark = SparkSession.builder.appName("responsi").getOrCreate()

print(spark)

In [9]:
import random
import pandas as pd
import json

random.seed(707)
n = 400
lokasi_list = ["Cabang Magelang", "Cabang Semarang", "Cabang Yogyakarta"]
kategori_dirty = ["laptop","Laptop","LAPTOP","hp","Hp","HP","tablet","Tablet"]

teknisi_ids = [f"TK{i:03d}" for i in range(1, 20)]
lokasi_per_teknisi = {kid: random.choice(lokasi_list) for kid in teknisi_ids}
                     
rows = []
for i in range(n):
    teknisi_id = random.choice(teknisi_ids)
    rows.append({
        "servis_id": f"SVS-{1000+i}",
        "teknisi_id": teknisi_id,
        "cabang": lokasi_per_teknisi[teknisi_id],
        "jenis_perangkat": random.choice(kategori_dirty),
        "durasi_servis_menit": random.randint(15, 120),
        "biaya_servis": random.choice(
        [75000,100000,150000,200000,350000]),
        "status_servis": random.choice(
        ["Selesai","Selesai","Selesai","Batal"]),
    })
    
rows += [rows[5], rows[24], rows[88]]
for idx in [7, 43, 148, 188]:
    rows[idx] = dict(rows[idx])
    rows[idx]["durasi_servis_menit"] = None
for idx in [11, 98]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_servis"] = None
for idx in [228, 278]:
    rows[idx] = dict(rows[idx])
    rows[idx]["durasi_servis_menit"] = -random.randint(1,10)
pd.DataFrame(rows).to_csv("servis.csv", index=False)
teknisi_master = [
    {"teknisi_id": kid, "nama_teknisi": f"Teknisi-{kid}",
    "cabang": lokasi_per_teknisi[kid],
    "rating_teknisi": round(random.uniform(3.0,5.0),1)}
    for kid in teknisi_ids
]
with open("teknisi.json", "w") as f:
    for k in teknisi_master:
        f.write(json.dumps(k) + "\n")
print("Data berhasil dibuat: servis.csv dan teknisi.json")

Data berhasil dibuat: servis.csv dan teknisi.json


In [10]:
print("===== SOAL NO 1 =====")

!hdfs dfs -mkdir -p /user/wildan/uts/raw/
!hdfs dfs -put -f servis.csv /user/wildan/uts/raw/

df_servis = spark.read.csv("hdfs://localhost:9000/user/wildan/uts/raw/", header=True, inferSchema=True)
df_teknisi = spark.read.json("teknisi.json")

print("\n===== SERVICE.CSV =====")
df_servis.printSchema()
print(f"Jumlah baris data pada data servis ada {df_servis.count()} baris data")

print("\n===== TEKNISI.JSON =====")
df_teknisi.printSchema()
print(f"Jumlah baris data pada data teknisi ada {df_teknisi.count()} baris data")

===== SOAL NO 1 =====



===== SERVICE.CSV =====
root
 |-- servis_id: string (nullable = true)
 |-- teknisi_id: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- jenis_perangkat: string (nullable = true)
 |-- durasi_servis_menit: double (nullable = true)
 |-- biaya_servis: double (nullable = true)
 |-- status_servis: string (nullable = true)

Jumlah baris data pada data servis ada 403 baris data

===== TEKNISI.JSON =====
root
 |-- cabang: string (nullable = true)
 |-- nama_teknisi: string (nullable = true)
 |-- rating_teknisi: double (nullable = true)
 |-- teknisi_id: string (nullable = true)

Jumlah baris data pada data teknisi ada 19 baris data


In [11]:
print("===== SOAL NO 2 =====")

count_before = df_servis.count()

print("\n===== PERMASALAHAN 1 =====")
df_servis.summary("count").show()
print("\nTerdapat data kosong di beberapa kolom tabel")

print("\n===== PERMASALAHAN 2 =====")
print(f"\nJumlah data yang identik/tidak duplikat ada {df_servis.dropDuplicates().count()} data")

print("\n===== PERMASALAHAN 3 =====")
df_servis.select("jenis_perangkat").distinct().show()
print("pada bagian tabel jenis perangkat terdapat inkonsistensi penulisan")

df_clean = df_servis.dropDuplicates()
df_clean = df_clean.dropna()
df_clean = df_clean.withColumn(("jenis_perangkat"), lower(col("jenis_perangkat")))

print("\n===== PENYELESAIAN PERMASALAHAN 1 =====")
df_clean.summary("count").show()
print("\nData yang sebelumnya kosong sekarang sudah dihapus menggunakan perintah dropna")

print("\n===== PENYELESAIAN PERMASALAHAN 2 =====")
print(f"\nJumlah data yang identik/tidak duplikat ada {df_clean.dropDuplicates().count()} data")

print("\n===== PENYELESAIAN PERMASALAHAN 3 =====")
df_clean.select("jenis_perangkat").distinct().show()
print("pada bagian tabel jenis perangkat sudah sama semua dengan semua huruf dibuat kecil semua dengan perintah lower")

count_after = df_clean.count()


print(f"\nTotal baris data servis.csv sebelum dibersihkan ada {count_before} baris data")
print(f"\nTotal baris data servis.csv setelah dibersihkan ada {count_after} baris data")

===== SOAL NO 2 =====

===== PERMASALAHAN 1 =====
+-------+---------+----------+------+---------------+-------------------+------------+-------------+
|summary|servis_id|teknisi_id|cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|
+-------+---------+----------+------+---------------+-------------------+------------+-------------+
|  count|      403|       403|   403|            403|                399|         401|          403|
+-------+---------+----------+------+---------------+-------------------+------------+-------------+


Terdapat data kosong di beberapa kolom tabel

===== PERMASALAHAN 2 =====

Jumlah data yang identik/tidak duplikat ada 400 data

===== PERMASALAHAN 3 =====
+---------------+
|jenis_perangkat|
+---------------+
|             HP|
|         tablet|
|         laptop|
|         Laptop|
|             hp|
|         Tablet|
|             Hp|
|         LAPTOP|
+---------------+

pada bagian tabel jenis perangkat terdapat inkonsistensi penulisan

===

In [13]:
print("===== SOAL NO 3 =====")

df_join = df_clean.join(df_servis.drop("cabang"),on="teknisi_id", how="left")

df_join.show(10)

===== SOAL NO 3 =====
+----------+---------+-----------------+---------------+-------------------+------------+-------------+---------+---------------+-------------------+------------+-------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|servis_id|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+---------+---------------+-------------------+------------+-------------+
|     TK013| SVS-1323|Cabang Yogyakarta|         laptop|               78.0|     75000.0|      Selesai| SVS-1374|             HP|               86.0|    350000.0|        Batal|
|     TK013| SVS-1323|Cabang Yogyakarta|         laptop|               78.0|     75000.0|      Selesai| SVS-1324|             HP|               35.0|    200000.0|        Batal|
|     TK013| SVS-1323|Cabang Yogyakarta|         laptop|               78.0|     75000.0|    

In [14]:
print("===== SOAL NO 4 =====")

df_target = df_join.Window.partitionBy("cabang").orderBy("status_servis", rank() == "Selesai")
df_rating = df_join.Window.partitionBy("rating").orderBy("rating_teknisi", row_number().over(4))

df_target.show()
df_rating.show()

===== SOAL NO 4 =====


AttributeError: 'DataFrame' object has no attribute 'Window'

In [15]:
print("===== SOAL NO 5 =====")

query_sql = spark.sql("""
    SELECT * FROM {df_clean} WHERE prsentase > target
    
""")

query_sql.show()

print("cabang yang perlu mendapatkan perhatian lebih dari manejemen adalah cabang .... karen cabang .... ")

===== SOAL NO 5 =====


ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '{'.(line 2, pos 18)

== SQL ==

    SELECT * FROM {df_clean} WHERE prsentase > target
------------------^^^


In [20]:
print("===== SOAL NO 6 =====")
path = ("hdfs://localhost:9000/user/wildan/uts/processed/laporan_servis")

df_parquet = df_join.withColumn("overwrite", col("cabang")).Parquet(path).count()

print(f"jumlah baris pada soal nomor 3 ada {df_join.count()} dan baris data yang di kirimkan ke hdfs dengan parquet ada {df_parquet} ")

===== SOAL NO 6 =====


AttributeError: 'DataFrame' object has no attribute 'Parquet'